In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.6 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/capstone_project")
DATA_DIR = PROJECT_DIR / "data"
YOLO_DIR = DATA_DIR / "yolo_particle_dataset"
RAW_DIR = DATA_DIR / "raw"

print("PROJECT_DIR:", PROJECT_DIR)
print("YOLO_DIR:", YOLO_DIR)
print("RAW_DIR:", RAW_DIR)

PROJECT_DIR: /content/drive/MyDrive/capstone_project
YOLO_DIR: /content/drive/MyDrive/capstone_project/data/yolo_particle_dataset
RAW_DIR: /content/drive/MyDrive/capstone_project/data/raw


In [ ]:
for p in [
    YOLO_DIR / "images" / "train",
    YOLO_DIR / "images" / "val",
    YOLO_DIR / "labels" / "train",
    YOLO_DIR / "labels" / "val",
]:
    print(p, "exists:", p.exists(), "count:", len(list(p.glob("*"))) if p.exists() else "MISSING")

/content/drive/MyDrive/capstone_project/data/yolo_particle_dataset/images/train exists: True count: 288
/content/drive/MyDrive/capstone_project/data/yolo_particle_dataset/images/val exists: True count: 72
/content/drive/MyDrive/capstone_project/data/yolo_particle_dataset/labels/train exists: True count: 288
/content/drive/MyDrive/capstone_project/data/yolo_particle_dataset/labels/val exists: True count: 72


In [ ]:
print(yaml_path.exists())
print(yaml_path.read_text())

True
path: /content/drive/MyDrive/capstone_project/data/yolo_particle_dataset
train: images/train
val: images/val

names:
  0: particle


In [ ]:
data_yaml = f"""
path: {YOLO_DIR.as_posix()}
train: images/train
val: images/val

names:
  0: particle
""".strip()

yaml_path = YOLO_DIR / "data.yaml"
yaml_path.write_text(data_yaml, encoding="utf-8")

print(yaml_path.read_text())

path: /content/drive/MyDrive/capstone_project/data/yolo_particle_dataset
train: images/train
val: images/val

names:
  0: particle


In [ ]:
import random

train_imgs = sorted((YOLO_DIR / "images" / "train").glob("*.jpg"))
sample_imgs = random.sample(train_imgs, min(5, len(train_imgs)))

for img_path in sample_imgs:
    lbl_path = YOLO_DIR / "labels" / "train" / (img_path.stem + ".txt")
    print(img_path.name, "-> label exists:", lbl_path.exists())
    if lbl_path.exists():
        print(lbl_path.read_text()[:300], "\n---")

vid_009_f001505.jpg -> label exists: True
0 0.6303693981899212 0.5898203592814368 0.07100866759631409 0.13373253493013937
0 0.598884422934952 0.8502994011976048 0.08574631558800207 0.12774451097804387
 
---
vid_010_f000960.jpg -> label exists: True
0 0.6163539587491684 0.5588822355289421 0.07507651363938798 0.13572854291417172
0 0.5909248170326015 0.3862275449101796 0.07749833666001343 0.13772455089820354
0 0.6012175648702595 0.9381237524950099 0.08839654025282756 0.12375249500997988
 
---
vid_010_f000405.jpg -> label exists: True
0 0.5636793080505657 0.47604790419161686 0.06417831004657362 0.12974051896207578
0 0.5285628742514968 0.3353293413173651 0.07628742514970041 0.12375249500998002
 
---
vid_008_f000550.jpg -> label exists: True
0 0.5867826391013596 0.4690618762475049 0.06780599385171229 0.0998003992015971
 
---
vid_008_f001715.jpg -> label exists: True
0 0.6259014817081164 0.6267465069860282 0.07562976237306379 0.13572854291417172
 
---


# **YOLO26n.pt**

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
results = model.train(
    data=str(yaml_path),
    epochs=80,
    imgsz=1024,
    batch=8,
    project=str(PROJECT_DIR / "runs"),
    name="particle_detect",
    pretrained=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/capstone_project/data/yolo_particle_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hs

In [ ]:
best_model_path = PROJECT_DIR / "runs" / "particle_detect" / "weights" / "best.pt"
print("Best model path:", best_model_path)

best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(yaml_path), imgsz=1024)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)

Best model path: /content/drive/MyDrive/capstone_project/runs/particle_detect/weights/best.pt
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.4 ms, read: 3.6±1.0 MB/s, size: 6.9 KB)
val: Scanning /content/drive/MyDrive/capstone_project/data/yolo_particle_dataset/labels/val.cache... 72 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 72/72 17.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.3it/s 3.8s
                   all         72        104       0.97      0.942      0.988       0.76
Speed: 18.1ms preprocess, 18.5ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /content/runs/detect/val
mAP50-95: 0.7596938346595521
mAP50: 0.9883112378595806


In [ ]:
video_path = RAW_DIR / "vid_008.mp4"

pred_results = best_model.predict(
    source=str(video_path),
    imgsz=1024,
    conf=0.25,
    save=True,
    project=str(PROJECT_DIR / "runs"),
    name="particle_predict_vid008"
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/2218) /content/drive/MyDrive/capstone_project/data/raw/vid_008.mp4: 768x1024 22 particles, 58.0ms
video 1/1 (frame 2/2218) /content/drive/MyDrive/capstone_project/data/raw/vid_008.mp4: 768x1024 22 particles, 16.4ms
video 1/1 (frame 3/2218) /content/drive/MyDrive/capstone_project/data/raw/vid_008.mp4: 768x1024 22 particles, 17.3ms
video 1/1 (frame 4/2218) /content/drive/MyDrive/capstone_project/data/raw/vid_008.mp4: 768x1024 22 particle

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/capstone_project")

MODEL_PATH = PROJECT_DIR / "runs" / "particle_detect" / "weights" / "best.pt"

print("Loading model from:", MODEL_PATH)

model = YOLO(str(MODEL_PATH))

Loading model from: /content/drive/MyDrive/capstone_project/runs/particle_detect/weights/best.pt


In [ ]:
VIDEO_PATH = PROJECT_DIR / "data" / "raw" / "vid_008.mp4"

model.predict(
    source=str(VIDEO_PATH),
    imgsz=1024,
    conf=0.25,
    save=True,
    project=str(PROJECT_DIR / "runs"),
    name="vid008_prediction",
    exist_ok=True
)

print("Prediction finished.")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/2218) /content/drive/MyDrive/capstone_project/data/raw/vid_008.mp4: 768x1024 22 particles, 58.1ms
video 1/1 (frame 2/2218) /content/drive/MyDrive/capstone_project/data/raw/vid_008.mp4: 768x1024 22 particles, 12.9ms
video 1/1 (frame 3/2218) /content/drive/MyDrive/capstone_project/data/raw/vid_008.mp4: 768x1024 22 particles, 12.8ms
video 1/1 (frame 4/2218) /content/drive/MyDrive/capstone_project/data/raw/vid_008.mp4: 768x1024 22 particle

In [ ]:
OUTPUT_DIR = PROJECT_DIR / "runs" / "vid008_prediction"
print("Saved results in:", OUTPUT_DIR)

Saved results in: /content/drive/MyDrive/capstone_project/runs/vid008_prediction


# **YOLO26s.pt**

In [ ]:
from ultralytics import YOLO

# YOLO26S
model_s = YOLO("yolo26s.pt")

results_s = model_s.train(
    data=str(yaml_path),
    epochs=80,
    imgsz=1024,
    batch=8,
    project=str(PROJECT_DIR / "runs"),
    name="particle_yolo26s",
    pretrained=True
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/capstone_project/data/yolo_particle_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hs

In [ ]:
last_checkpoint_s = PROJECT_DIR / "runs" / "particle_yolo26s" / "weights" / "last.pt"
print("Resuming from:", last_checkpoint_s)

Resuming from: /content/drive/MyDrive/capstone_project/runs/particle_yolo26s/weights/last.pt


In [ ]:
from ultralytics import YOLO
model_s_resume = YOLO(str(last_checkpoint_s))

results_s_resume = model_s_resume.train(
    data=str(yaml_path),
    epochs=80,   # total target epochs
    imgsz=1024,
    batch=8,
    project=str(PROJECT_DIR / "runs"),
    name="particle_yolo26s",
    resume=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/capstone_project/data/yolo_particle_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hs

In [ ]:
best_model_path_s = PROJECT_DIR / "runs" / "particle_yolo26s" / "weights" / "best.pt"
print("YOLO26S best model:", best_model_path_s)

model_s_best = YOLO(str(best_model_path_s))

metrics_s = model_s_best.val(data=str(yaml_path), imgsz=1024)

print("YOLO26S mAP50-95:", metrics_s.box.map)
print("YOLO26S mAP50:", metrics_s.box.map50)

YOLO26S best model: /content/drive/MyDrive/capstone_project/runs/particle_yolo26s/weights/best.pt
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.3 ms, read: 5.0±0.5 MB/s, size: 6.9 KB)
val: Scanning /content/drive/MyDrive/capstone_project/data/yolo_particle_dataset/labels/val.cache... 72 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 72/72 17.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.0it/s 4.9s
                   all         72        104       0.95      0.917      0.972      0.752
Speed: 22.2ms preprocess, 22.2ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/runs/detect/val
YOLO26S mAP50-95: 0.7517531008954138
YOLO26S mAP50: 0.9724535757220532


# **YOLO26m.pt**

In [ ]:
from ultralytics import YOLO

# YOLO26M
model_m = YOLO("yolo26m.pt")

results_m = model_m.train(
    data=str(yaml_path),
    epochs=80,
    imgsz=1024,
    batch=8,
    project=str(PROJECT_DIR / "runs"),
    name="particle_yolo26m",
    pretrained=True
)

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/capstone_project/data/yolo_particle_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=particle_yolo26m, nbs=64, nms=False, opset=None, optimize=False, op